In [1]:
# --- Imports  All Required Libraries
import os
import shutil
import re
from tqdm import tqdm
import PyPDF2
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import chromadb

# --- Setup Configuration 
CONFIG = {
    "CHUNK_SIZE": 400,
    "CHUNK_OVERLAP": 100,
    "TOP_K": 5,
    "SIMILARITY_THRESHOLD": 0.7,
    "DOCS_FOLDER": "./docs",
    "VECTOR_DB_FOLDER": "./vector_db",
    "MAX_INPUT_TOKENS": 1500,
    "EMBED_MODEL": "all-MiniLM-L6-v2",
    "QA_MODEL": "google/flan-t5-small"
}

# Clear previous vector DB from the vector database 
def reset_vector_db(path):
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)

# Load PDF from the docs folder  
def load_pdf_text(path):
    """Extracts text from a PDF file."""
    text = ""
    try:
        with open(path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                text += page.extract_text() or ""
    except Exception as e:
        print(f"Error reading {path}: {e}")
    return text

# --- Load all PDF documents and extract their text and prepare text for processing
def load_all_documents(folder):
    """Loads and returns a dictionary of PDF texts keyed by filename."""
    docs = {}
    for file in os.listdir(folder):
        if file.lower().endswith(".pdf"):
            txt = load_pdf_text(os.path.join(folder, file))
            if txt.strip():
                docs[file] = txt
            else:
                print(f"Warning: {file} has no text.")
    print(f"Loaded {len(docs)} documents.")
    return docs

# --- Chunking the text into smaller overlapping pieces
def chunk_text(text, size=CONFIG["CHUNK_SIZE"], overlap=CONFIG["CHUNK_OVERLAP"]):
    """Splits text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + size)
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

def create_corpus(all_docs):
    """Creates a chunked corpus from documents."""
    corpus = []
    for doc_name, text in all_docs.items():
        for i, ch in enumerate(chunk_text(text)):
            corpus.append({"doc": doc_name, "chunk_id": i, "text": ch})
    print(f"Created {len(corpus)} chunks from all documents.")
    return corpus

# --- Convert document chunks into vector embeddings and store them for search
def setup_vector_db(corpus, embed_model_name=CONFIG["EMBED_MODEL"], db_path=CONFIG["VECTOR_DB_FOLDER"]):
    """Embeds corpus and inserts into Chroma vector DB."""
    embed_model = SentenceTransformer(embed_model_name)
    client = chromadb.PersistentClient(path=db_path)
    collection = client.get_or_create_collection(name="rag_collection")

    for item in tqdm(corpus, desc="Indexing"):
        try:
            emb = embed_model.encode(item["text"]).tolist()
            uid = f"{item['doc']}_chunk{item['chunk_id']}"
            collection.add(
                documents=[item["text"]],
                embeddings=[emb],
                ids=[uid],
                metadatas=[{"doc": item["doc"], "chunk": item["chunk_id"]}],
            )
        except Exception as e:
            print(f"Embedding error for {uid}: {e}")
    print("All chunks inserted into Chroma.")
    return embed_model, collection


# --- Prepare the knowledge base for retrieval by encoding text into vectors
def retrieve(query, embed_model, collection, top_k=CONFIG["TOP_K"]):
    """Retrieve top-k relevant chunks from vector DB."""
    query_emb = embed_model.encode(query).tolist()
    results = collection.query(query_embeddings=[query_emb], n_results=top_k)
    docs = []
    ids = results.get("ids", [[]])[0]
    if not ids:
        return docs
    for i in range(len(ids)):
        docs.append({
            "id": ids[i],
            "text": results.get("documents", [[]])[0][i],
            "score": results.get("distances", [[]])[0][i],
            "metadata": results.get("metadatas", [[]])[0][i]
        })
    return docs

# --- Prepare the LLM input by combining the user’s question with relevant document chunks
def build_prompt(question, retrieved_docs, max_tokens=CONFIG["MAX_INPUT_TOKENS"]):
    """Builds LLM prompt with retrieved context."""
    context = ""
    token_count = 0
    for d in retrieved_docs:
        chunk_tokens = len(d["text"].split())
        if token_count + chunk_tokens > max_tokens:
            break
        sentences = d["text"].split(". ")
        bullet_text = "\n• " + "\n• ".join([s.strip() for s in sentences if s.strip()])
        context += f"[Source: {d['metadata']['doc']}#chunk{d['metadata']['chunk']}] {bullet_text}\n"
        token_count += chunk_tokens
    prompt = f"Answer the question using only the sources given. Cite sources as bullet points.\n\nQuestion: {question}\n\nSources:\n{context}\nAnswer:"
    return prompt

# --- Answer Formate of Final ---
def clean_and_format_answer(raw_text):
    """Cleans and categorizes LLM output into Safety, Features, Warnings."""
    text = re.sub(r"\d+", "", raw_text)
    text = re.sub(r"\s+", " ", text).strip()

    sentences = re.split(r"\. |\n|•", text)
    seen = set()
    cleaned_lines = []
    for s in sentences:
        s_clean = s.strip()
        if len(s_clean) > 15 and s_clean.lower() not in seen:
            s_clean = s_clean[0].upper() + s_clean[1:]
            cleaned_lines.append(s_clean)
            seen.add(s_clean.lower())

    # Categorize bullets points into sections
    safety_keywords = ["do not", "follow", "ensure", "exercise caution"]
    features_keywords = ["design", "feature", "performance", "reliable", "encapsulated", "lifted"]
    warnings_keywords = ["hazardous", "important notice", "dangerous"]

    safety_lines = [f"• {line}." for line in cleaned_lines if any(k in line.lower() for k in safety_keywords)]
    feature_lines = [f"• {line}." for line in cleaned_lines if any(k in line.lower() for k in features_keywords)]
    warning_lines = [f"• {line}." for line in cleaned_lines if any(k in line.lower() for k in warnings_keywords)]

    # Remove duplicates across sections 
    safety_set = set(safety_lines)
    feature_lines = [f for f in feature_lines if f not in safety_set]
    feature_set = set(feature_lines)
    warning_lines = [w for w in warning_lines if w not in safety_set and w not in feature_set]

    output = ""
    if safety_lines:
        output += "### Safety Precautions:\n" + "\n".join(safety_lines) + "\n\n"
    if feature_lines:
        output += "### Key Features:\n" + "\n".join(feature_lines) + "\n\n"
    if warning_lines:
        output += "### Warnings:\n" + "\n".join(warning_lines)

    if not output.strip():
        output = "\n".join([f"• {line}." for line in cleaned_lines])

    return output.strip()

# --- Answer Generator Function
def generate_answer(question, embed_model, collection):
    """Retrieve relevant chunks, generate LLM answer, and format it."""
    retrieved = retrieve(question, embed_model, collection)
    if not retrieved:
        return "Sorry, no relevant information found."
    filtered = [d for d in retrieved if d["score"] <= CONFIG["SIMILARITY_THRESHOLD"]] or retrieved
    prompt = build_prompt(question, filtered)
    qa_pipeline = pipeline("text2text-generation", model=CONFIG["QA_MODEL"], tokenizer=CONFIG["QA_MODEL"])
    result = qa_pipeline(prompt, max_new_tokens=300, do_sample=False)[0]["generated_text"]
    return clean_and_format_answer(result)

# --- Main Execution Flow
if __name__ == "__main__":
    reset_vector_db(CONFIG["VECTOR_DB_FOLDER"])
    docs = load_all_documents(CONFIG["DOCS_FOLDER"])
    corpus = create_corpus(docs)
    embed_model, collection = setup_vector_db(corpus)

    # Test question and generate answer
    question = "What are the safety precautions and features of cyclone?"
    answer = generate_answer(question, embed_model, collection)
    print("Question:", question)
    print("Answer:\n", answer)
    
    # --- Interactive Q&A  contineously if u want enable this code to ?: ask question like eg: what is cyclone? ---
_ = """

while True:
    q = input("\nEnter your question (or 'quit'): ")
    if q.lower() == "quit":
        break
    print("Answer:\n", generate_answer(q, embed_model, collection))
    
    
"""

Loaded 11 documents.
Created 934 chunks from all documents.


Indexing: 100%|██████████| 934/934 [00:46<00:00, 20.28it/s]


All chunks inserted into Chroma.


Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (597 > 512). Running this sequence through the model will result in indexing errors


Question: What are the safety precautions and features of cyclone?
Answer:
 ### Safety Precautions:
• Do not attempt to install, operate or service your new Cyclone without proper instruction and until you have been thoroughly trained in its use by your employer.
• Follow the instructions in this manual to ensure not only ease of use, but also optimum use and longevity of the equipment.
• Do not operate the equipment until you are thoroughly familiar with all the instructions, prohibitions and recommendations contained in.

### Key Features:
• The unit must be lifted by a means with sufficient lifting capacity.
• The cyclone is fully encapsulated if properly connected during installation.

### Warnings:
• Or cyclone installed in hazardous a r e a s.
• IMPORTANT NOTICESPlease read these instructions carefully.


##  Simple Evaluation Metrics for single query not all (Optional)
- just for undesrstanding purpose if want u can Build Seperate  all queries Evaluation and save it  in .csv formate for future


In [ ]:
# --- Simple Evaluation Metrics ---
from sklearn.metrics import precision_score, recall_score, f1_score

def tokenize(text):
    """Simple tokenizer: lowercase and split by whitespace"""
    return set(text.lower().split())

def evaluate_answer(predicted, reference):
    """
    Evaluate the predicted answer against a reference answer.
    Returns precision, recall, and F1-score (token-level).
    """
    pred_tokens = tokenize(predicted)
    ref_tokens = tokenize(reference)
    
    if not ref_tokens:
        return 0.0, 0.0, 0.0

    true_positives = len(pred_tokens & ref_tokens)
    precision = true_positives / len(pred_tokens) if pred_tokens else 0
    recall = true_positives / len(ref_tokens) if ref_tokens else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# --- Example usage 
reference_answer = """
• Ensure proper protective equipment is worn.
• Follow standard operating procedures.
• The cyclone is designed to safely handle exhaust gases.
"""

precision, recall, f1 = evaluate_answer(answer, reference_answer)
print(f"\nEvaluation Metrics:\nPrecision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1:.2f}")



Evaluation Metrics:
Precision: 0.11, Recall: 0.47, F1-score: 0.17
